# Data Transformation

> **Chapter 1.4 — Data Exploration**

After cleaning, your features are *correct* — but they are rarely *comparable*. Age lives in the tens, salary in the tens of thousands, and experience may be a skewed count of years. Left as-is, the large-scale feature drowns out the others in any algorithm that measures distance or sums weighted inputs.

**Data transformation** rescales features onto comparable ranges so that each one contributes fairly. In this notebook you will:

- Understand **why** transformation is necessary — and which algorithms demand it
- Apply **Normalization** (Min–Max scaling) to squeeze features into `[0, 1]`
- Apply **Standardization** (Z-score) to center features at mean 0, std 1
- Apply **Robust scaling** to resist outliers
- See, with detailed matplotlib graphics, exactly **how** each transform reshapes a distribution and **when** to reach for each one


---

## 1. Why Transform?

Many models are **scale-sensitive**: they implicitly assume every feature spans a similar range. When one feature is numerically huge, it dominates the math regardless of how informative it actually is.

::::{grid} 1 2 2 2
:gutter: 3

:::{grid-item-card} ⚠️ Scale-sensitive — must transform
:class-header: sd-bg-danger sd-text-white

^^^
- **Distance-based** — k-NN, k-Means, SVM (RBF)
- **Gradient-based** — linear/logistic regression, neural networks
- **Regularized** — Ridge, Lasso (penalty depends on coefficient size)
- **PCA** — maximizes variance, so the largest-scale feature wins
:::

:::{grid-item-card} ✅ Scale-invariant — transform optional
:class-header: sd-bg-success sd-text-white

^^^
- **Tree-based** — decision trees, random forests, gradient boosting
- These split on thresholds *within* a single feature, so monotonic rescaling changes nothing.
:::
::::

```{admonition} The core idea
:class: note
A transformation that is **monotonic** (preserves order) never changes the *information* in a feature — only its *units*. Normalization and standardization are both linear, so they preserve the shape of the distribution exactly; they just move and stretch the axis.
```


---

## 2. A Dataset on Three Different Scales

Let's build a small dataset whose three features differ wildly in both **scale** and **shape** — exactly the situation transformation is designed to fix.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 500

age        = rng.normal(38, 8, n)            # tens         — roughly symmetric
salary     = rng.normal(60_000, 18_000, n)   # tens of thousands
experience = rng.gamma(2.0, 2.0, n)          # 0–~15        — right-skewed

df = pd.DataFrame({"age": age, "salary": salary, "experience": experience})
print(df.describe().round(2))

Notice the `mean` and `std` rows: `salary` is roughly **1,500×** larger than `age`. Any algorithm that adds these numbers together will effectively ignore age and experience.


In [ ]:
# A boxplot on a shared axis makes the problem obvious: salary flattens everything else.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([df["age"], df["experience"], df["salary"]],
                labels=["age", "experience", "salary"], patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.5))
axes[0].set_title("Raw features, shared axis — salary dominates")
axes[0].set_ylabel("value")

# Same data, log scale — now we can actually see all three.
axes[1].boxplot([df["age"], df["experience"], df["salary"]],
                labels=["age", "experience", "salary"], patch_artist=True,
                boxprops=dict(facecolor="seagreen", alpha=0.5))
axes[1].set_yscale("log")
axes[1].set_title("Same data on a log axis — the others reappear")
axes[1].set_ylabel("value (log scale)")

plt.tight_layout()
plt.show()

### How scale corrupts a distance calculation

Distance-based models compare rows using Euclidean distance:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

Because the differences are **squared**, the feature with the largest numbers dominates the sum. The cell below computes the distance between two employees using each feature's raw contribution.


In [ ]:
a, b = df.iloc[0], df.iloc[1]
diffs = (a - b) ** 2

print("Squared contribution of each feature to the distance between row 0 and row 1:")
for col in df.columns:
    print(f"  {col:<11}: {diffs[col]:,.2f}")

total = diffs.sum()
print(f"\n  salary accounts for {diffs['salary'] / total:.1%} of the total distance —")
print("  age and experience are effectively ignored.")

---

## 3. Normalization (Min–Max Scaling)

**Normalization** linearly rescales a feature into a fixed range, almost always $[0, 1]$:

$$\boxed{x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}}$$

The minimum maps to 0, the maximum to 1, and everything else lands proportionally in between.

| Property | Effect |
|---|---|
| **Range** | Guaranteed $[0, 1]$ |
| **Shape** | Unchanged (linear transform) |
| **Outliers** | **Very sensitive** — a single extreme value compresses everyone else into a tiny band |
| **Use when** | You need a bounded range (e.g. image pixels, neural-network inputs) and data has few outliers |


In [ ]:
def min_max(s: pd.Series) -> pd.Series:
    # Scale a series into [0, 1]
    return (s - s.min()) / (s.max() - s.min())

df_norm = df.apply(min_max)
print(df_norm.describe().round(3))
print("\nEvery column now has min = 0 and max = 1.")

In [ ]:
# Before vs after: the SHAPE of each distribution is identical — only the x-axis changed.
fig, axes = plt.subplots(3, 2, figsize=(12, 9))
colors = ["steelblue", "seagreen", "indianred"]

for row, (col, c) in enumerate(zip(df.columns, colors)):
    axes[row, 0].hist(df[col], bins=30, color=c, alpha=0.7, edgecolor="white")
    axes[row, 0].set_title(f"{col} — raw  (range {df[col].min():.0f} … {df[col].max():.0f})")

    axes[row, 1].hist(df_norm[col], bins=30, color=c, alpha=0.7, edgecolor="white")
    axes[row, 1].set_title(f"{col} — normalized  (range 0 … 1)")
    axes[row, 1].set_xlim(-0.05, 1.05)

fig.suptitle("Min–Max Normalization: identical shapes, common [0, 1] range", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---

## 4. Standardization (Z-score)

**Standardization** recenters a feature on mean 0 and rescales it to unit standard deviation:

$$\boxed{x' = \frac{x - \mu}{\sigma}}$$

The result has **no fixed range**, but it answers a universal question: *how many standard deviations is this value from the mean?* A standardized value of `+2` means "two standard deviations above average," whatever the original units.

| Property | Effect |
|---|---|
| **Center / spread** | mean = 0, std = 1 |
| **Range** | Unbounded (typically about −3 … +3 for normal data) |
| **Shape** | Unchanged (linear transform) |
| **Outliers** | Less sensitive than Min–Max, but still affected (they inflate $\sigma$) |
| **Use when** | The algorithm assumes roughly centered data — regression, PCA, neural nets — and you don't need a hard range |


In [ ]:
def z_score(s: pd.Series) -> pd.Series:
    # Center on mean 0, scale to std 1
    return (s - s.mean()) / s.std()

df_std = df.apply(z_score)
print(df_std.describe().round(3))
print("\nEvery column now has mean ≈ 0 and std ≈ 1.")

In [ ]:
# Overlay all three standardized features: now directly comparable on one axis.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: raw, can't share a sensible axis
for col, c in zip(df.columns, colors):
    axes[0].hist(df[col], bins=30, alpha=0.5, label=col, color=c)
axes[0].set_title("Raw — incomparable, salary off the chart")
axes[0].legend()

# Right: standardized, all centered at 0 with the same spread
for col, c in zip(df.columns, colors):
    axes[1].hist(df_std[col], bins=30, alpha=0.5, label=col, color=c)
axes[1].axvline(0, color="black", lw=1, ls="--")
axes[1].set_title("Standardized — all centered at 0, std 1")
axes[1].set_xlabel("standard deviations from the mean")
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 5. Robust Scaling — Surviving Outliers

Both methods above use statistics that **outliers corrupt**: Min–Max uses the min and max (which *are* the outliers), and standardization uses the mean and std (both pulled by extremes). **Robust scaling** swaps them for the median and the IQR — quantities that barely move when a few points go haywire:

$$\boxed{x' = \frac{x - \text{median}}{IQR} = \frac{x - Q_2}{Q_3 - Q_1}}$$

Let's prove it. We inject one absurd salary and compare how each method reacts.


In [ ]:
def robust(s: pd.Series) -> pd.Series:
    # Center on the median, scale by the IQR — outlier-resistant
    q1, q3 = s.quantile([0.25, 0.75])
    return (s - s.median()) / (q3 - q1)

# Inject one extreme outlier: a $5,000,000 salary
salary_out = df["salary"].copy()
salary_out.iloc[0] = 5_000_000

comparison = pd.DataFrame({
    "min_max":  min_max(salary_out),
    "z_score":  z_score(salary_out),
    "robust":   robust(salary_out),
})

print("Spread of the 499 NON-outlier points after each transform:")
normal = comparison.iloc[1:]
print(normal.describe().loc[["min", "25%", "50%", "75%", "max"]].round(3))

In [ ]:
# Visualize how the 499 normal points are distributed after each transform.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
methods = [("min_max", "Min–Max", "indianred"),
           ("z_score", "Standardization", "steelblue"),
           ("robust",  "Robust", "seagreen")]

for ax, (key, title, c) in zip(axes, methods):
    ax.hist(comparison[key].iloc[1:], bins=30, color=c, alpha=0.75, edgecolor="white")
    ax.set_title(f"{title}\n(normal points only)")
    ax.set_xlabel("scaled value")

fig.suptitle("One $5M outlier — Min–Max crushes the rest into a sliver; "
             "robust scaling keeps them spread out", y=1.03, fontsize=12)
plt.tight_layout()
plt.show()

Min–Max squashes all 499 legitimate salaries into a tiny band near 0 because the lone $5M value defines the top of the range. Standardization is better but still compressed (the outlier inflated $\sigma$). **Robust scaling** is barely affected — the median and IQR ignored the outlier entirely.


---

## 6. All Three, Side by Side

Using the clean `salary` feature (no injected outlier), here is the same data under each transform. The **shape never changes** — only the location and width of the axis.


In [ ]:
salary = df["salary"]
transforms = {
    "Raw":             salary,
    "Normalized [0,1]": min_max(salary),
    "Standardized":     z_score(salary),
    "Robust":           robust(salary),
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (name, data) in zip(axes, transforms.items()):
    ax.hist(data, bins=30, color="slateblue", alpha=0.75, edgecolor="white")
    ax.set_title(name)
    ax.axvline(data.mean(), color="red", lw=1, ls="--")

fig.suptitle("Same salary distribution under four scalings — shape preserved, axis changes",
             y=1.04, fontsize=12)
plt.tight_layout()
plt.show()

---

## 7. Which One Should I Use?

```{list-table}
:header-rows: 1
:widths: 22 30 48

* - Method
  - Output
  - Reach for it when…
* - **Normalization** (Min–Max)
  - Bounded `[0, 1]`
  - You need a hard range (image pixels, neural-net inputs) **and** outliers are already handled
* - **Standardization** (Z-score)
  - mean 0, std 1
  - The default for most models — regression, SVM, PCA, neural nets — when data is roughly bell-shaped
* - **Robust scaling**
  - median 0, IQR 1
  - Outliers remain in the data and you can't or don't want to remove them
```

```{admonition} Standardization ≠ a normal distribution
:class: warning
Standardizing makes the **mean 0 and std 1** — it does **not** make a skewed feature symmetric. To change the *shape* of a distribution (e.g. fix right-skew) you need a non-linear transform such as `np.log1p(x)` or a Box–Cox / Yeo–Johnson power transform. Scaling and reshaping are different jobs.
```


In [ ]:
# Demonstration: a log transform reshapes the skewed 'experience' feature;
# standardization alone would only shift and stretch it.
exp = df["experience"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(exp, bins=30, color="indianred", alpha=0.75, edgecolor="white")
axes[0].set_title(f"experience — raw (skew = {exp.skew():.2f})")

axes[1].hist(z_score(exp), bins=30, color="steelblue", alpha=0.75, edgecolor="white")
axes[1].set_title(f"standardized (skew = {z_score(exp).skew():.2f}) — still skewed")

log_exp = np.log1p(exp)
axes[2].hist(log_exp, bins=30, color="seagreen", alpha=0.75, edgecolor="white")
axes[2].set_title(f"log1p (skew = {log_exp.skew():.2f}) — now symmetric")

fig.suptitle("Scaling moves and stretches; only a non-linear transform reshapes",
             y=1.03, fontsize=12)
plt.tight_layout()
plt.show()

---

## 8. The Cardinal Rule: Fit on Train, Apply to Test

Every transform depends on statistics — min, max, mean, std, median, IQR. These must be computed **only on the training set**. If you compute them over the full dataset before splitting, information from the test set leaks into training and your evaluation becomes optimistic and untrustworthy.

```{admonition} Data leakage
:class: danger
**Wrong:** scale the whole dataset, *then* split into train/test.
**Right:** split first → compute the scaling statistics on **train** → apply those *same* statistics to **test**.
```


In [ ]:
# Correct workflow, written out explicitly (no sklearn needed).
split = int(0.8 * len(df))
train, test = df.iloc[:split], df.iloc[split:]

# 1. Learn the statistics from TRAIN only
mu, sigma = train.mean(), train.std()

# 2. Apply the SAME statistics to both sets
train_scaled = (train - mu) / sigma
test_scaled  = (test  - mu) / sigma

print("Train is centered exactly (it defined the statistics):")
print(train_scaled.mean().round(3).to_string())
print("\nTest is close to — but not exactly — 0, which is correct and honest:")
print(test_scaled.mean().round(3).to_string())

---

## Exercises

```{admonition} Exercise 1 — Reverse the transform
:class: tip
Normalization is invertible. Given a normalized value `x'` and the original `min`/`max`, write a function that recovers the original `x`. Verify it round-trips the `age` column.
```

```{admonition} Exercise 2 — Distance, fixed
:class: tip
Recompute the squared per-feature distance contributions from Section 2, but on the **standardized** data. Confirm that no single feature dominates anymore.
```

```{admonition} Exercise 3 — Pick the right tool
:class: tip
The `salary` column has a few legitimate high earners (not errors). You need to feed it to a k-NN model. Which transform would you choose, and why? Plot the result to support your answer.
```

````{admonition} Exercise 4 — MaxAbs scaling
:class: tip
Another scaler, **MaxAbs**, divides by the maximum absolute value: `x / max(|x|)`, mapping data to `[-1, 1]` while preserving zeros (useful for sparse data). Implement it and compare it to Min–Max on the `age` column.
````


---

## Summary

::::{grid} 1 2 2 2
:gutter: 3

:::{grid-item-card} What We Covered
:class-header: sd-bg-primary sd-text-white

^^^
| Transform | Formula | Output |
|---|---|---|
| Normalization | $(x - \min)/(\max - \min)$ | $[0, 1]$ |
| Standardization | $(x - \mu)/\sigma$ | mean 0, std 1 |
| Robust | $(x - Q_2)/IQR$ | median 0, IQR 1 |
:::

:::{grid-item-card} Key Takeaways
:class-header: sd-bg-success sd-text-white

^^^
- Transform for **distance- and gradient-based** models; trees don't need it
- Scaling is **linear** — it never changes a distribution's *shape*
- Use **robust** scaling when outliers remain
- To fix **skew**, use a non-linear transform (log, Box–Cox)
- Always **fit on train, apply to test**
:::
::::

```{admonition} What's Next
:class: note
With your data cleaned and transformed, it is finally model-ready. The next chapters use these prepared features for regression, classification, and feature engineering.
```
